## Hybrid Retrieval Tuning — BM25 × Bi-Encoder × Cross-Encoder

Sweep all three components (all grids live in `configs/hybrid-tuning/hybrid_sweep.yaml`):

| component | grid |
|---|---|
| BM25 | `k1 × b` |
| Bi-encoder (dense) | trained vs pretrained |
| Cross-encoder (rerank) | trained vs pretrained |
| Fusion | RRF `w_bm25 × k` |

Index ONCE (cached); fusion grid re-fuses cached rankings (instant). Batch sizes are
VRAM-auto (`vnlegal_rag_v2.utils.batch_size`, same source as `update_batch_size.py`).

Edit the config, then run the two cells below. Results → `results/hybrid-tuning/scores.json`.


In [1]:
import os
from getpass import getpass
os.environ["GH_TOKEN"] = getpass("Paste GitHub token: ")
os.environ["HF_TOKEN"] = getpass("Paste Hugging Face token: ")

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
!rm -rf vnlegal-rag-v2
!git clone https://${GH_TOKEN}@github.com/hyperformancelabs/vnlegal-rag-v2.git
%cd vnlegal-rag-v2
%pip install .

/content
Cloning into 'vnlegal-rag-v2'...
remote: Enumerating objects: 837, done.
remote: Counting objects: 100% (376/376), done.
remote: Compressing objects: 100% (258/258), done.
remote: Total 837 (delta 190), reused 275 (delta 100), pack-reused 461 (from 1)
Receiving objects: 100% (837/837), 121.68 MiB | 18.86 MiB/s, done.
Resolving deltas: 100% (432/432), done.
/content/vnlegal-rag-v2
Processing /content/vnlegal-rag-v2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 132.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 25.7 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 137.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [4]:
# Re-run after pushing code/config changes
!git pull
%pip install .

Already up to date.
Processing /content/vnlegal-rag-v2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for vnlegal-rag-v2: filename=vnlegal_rag_v2-0.1.0-py3-none-any.whl size=27588 sha256=9efc310250c65b25e390377ed5006722772219792ec126c6a98ad4d4a01404c7
  Stored in directory: /root/.cache/pip/wheels/86/8f/16/c3124b4be639db7d2b065a5b714f123859ed3325b439347cad
Successfully built vnlegal-rag-v2
  Attempting uninstall: vnlegal-rag-v2
    Found existing installation: vnlegal-rag-v2 0.1.0
    Uninstalling vnlegal-rag-v2-0.1.0:
      Successfully uninstalled vnlegal-rag-v2-0.1.0


In [5]:
# Download processed data (incl. pre-segmented corpus_pyvi.csv / eval_pyvi.csv)
!gdown 1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
!mkdir -p data/processed
!unzip -o data_processed.zip -d data/processed
!rm -rf data/processed/__MACOSX data_processed.zip

Downloading...
From (original): https://drive.google.com/uc?id=1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm
From (redirected): https://drive.google.com/uc?id=1vkAnfAHNPEhu-aKrhR17RbyY3TYqXLzm&confirm=t&uuid=6cc58be8-2d3c-4ffc-b53e-4dc58ce4fb53
To: /content/vnlegal-rag-v2/data_processed.zip
100% 360M/360M [00:02<00:00, 152MB/s]  
Archive:  data_processed.zip
  inflating: data/processed/corpus.csv  
  inflating: data/processed/corpus_pyvi.csv  
  inflating: data/processed/__MACOSX/._corpus_pyvi.csv  
  inflating: data/processed/train.csv  
  inflating: data/processed/eval.csv  
  inflating: data/processed/train_pyvi.csv  
  inflating: data/processed/eval_pyvi.csv  
  inflating: data/processed/train_underthesea.csv  
  inflating: data/processed/eval_underthesea.csv  
  inflating: data/processed/corpus_underthesea.csv  


In [6]:
!pip install -U huggingface_hub
!hf auth login --token ${HF_TOKEN}

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 48.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.18.0
    Uninstalling huggingface_hub-1.18.0:
      Successfully uninstalled huggingface_hub-1.18.0
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `base` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## Run the sweep

All grids (models, bm25 k1/b, weights, fusion_k, candidate sets) are in the config below.
Edit it to add/swap models or narrow the grid.


In [7]:
!cat configs/hybrid-tuning/hybrid_sweep.yaml

name: hybrid-tuning

data:
  data_path: data/processed
  corpus_path: data/processed/corpus_pyvi.csv   # pre-segmented (pyvi)
  eval_file: eval_pyvi.csv                       # pre-segmented (pyvi)

# All three components are LOCKED from prior sweeps — only the FUSION is tuned:
#   dense  trained   0.6372 >> pretrained 0.4615   (dense-selection)
#   cross  trained   0.7725  (ref 0.7754)           (cross-encoder train)
#   bm25   k1=1.0 b=0.9 pyvi  0.3928 best           (baseline-selection)
models:
  dense:
    trained: phatvucoder/vietnamese-bi-encoder
  cross:
    trained: phatvucoder/PhoRanker-legal-vn

# ── Phase 1: retrieval grid — dense + BM25 fixed; tune fusion only ───────
retrieval:
  top_k: 100
  dense: [trained]
  bm25:
    variant: bm25
    segmentation: pyvi                  # metadata only; corpus file is already segmented
    k1: 1.0                             # fixed — baseline best
    b: 0.9                              # fixed — baseline best
  w_bm25: [0.0, 0.1, 0.2

In [8]:
# VRAM-aware batch sizes (prints report; tune_hybrid uses the same logic internally)
!python scripts/update_batch_size.py --vram auto

Target: 23GB VRAM

  Model                                            |  enc | rerk | trBi |  trX | cap
  ------------------------------------------------ | ---- | ---- | ---- | ---- | ---
  multilingual-e5-base                             |  128 |   64 |   64 |   32 | 128
  vietnamese-bi-encoder                            |  256 |  128 |  128 |   64 | 256
  granite-embedding-97m-multilingual-r2            |  128 |   64 |   64 |   32 | 128
  embeddinggemma-300m                              |   64 |   32 |   16 |    8 | 64
  PhoRanker                                        |  256 |  128 |  128 |   64 | 256
  gte-multilingual-reranker-base                   |  128 |   64 |   64 |   32 | 128


configs/dense-selection/:
  e5-base.yaml: encode → bs=128 (already set)
  embeddinggemma-300m.yaml: encode → bs=64 (already set)
  granite-97m.yaml: encode → bs=128 (already set)
  vietnamese-bi-encoder.yaml: encode → bs=256 (already set)

configs/model-selection/:
  embeddinggemma-300m.yaml: encode

In [9]:
!python scripts/tune_hybrid.py configs/hybrid-tuning/hybrid_sweep.yaml

VRAM=23GB → encode bs=256, rerank bs=128

corpus: 262,168 docs | queries: 13,364

dense[trained]: encoding corpus…
Batches: 100% 1025/1025 [26:47<00:00,  1.57s/it]
Batches: 100% 53/53 [00:09<00:00,  5.52it/s]
dense[trained]: built + cached
Building index from IDs objects
bm25 k1=1.0 b=0.9: built + cached

Phase 1: 66 retrieval combos logged


Phase 2: reranking top-1 config(s) by success@100:
  → auto-trained-w=0.3-fk=10
rerank auto-trained-w=0.3-fk=10 × trained: running…
Reranking (Cross-Encoder): 100% 10440/10440 [2:45:28<00:00,  1.05it/s] 
rerank auto-trained-w=0.3-fk=10 × trained: done + cached

=== Retrieval — top 15 by success@100 ===
combo                                          success@100     mrr@10
----------------------------------------------------------------------
retr_trained_k1=1.0_b=0.9_w=0.3_fk=10              0.9811     0.6177
retr_trained_k1=1.0_b=0.9_w=0.3_fk=5               0.9808     0.6284
retr_trained_k1=1.0_b=0.9_w=0.3_fk=20              0.9808     0.6009
ret

## Results

In [17]:
!cat results/hybrid-tuning/scores.json | python3 -m json.tool | tail -60

        "ndcg@5": 0.38949772026840507,
        "ndcg@10": 0.4199071315017222,
        "ndcg@20": 0.4409343033063253,
        "success@100": 0.8525142173002095,
        "stage": "retrieval",
        "params": {
            "dense": "trained",
            "k1": 1.0,
            "b": 0.9,
            "w_bm25": 1.0,
            "w_dense": 0.0,
            "fusion_k": 60,
            "top_k": 100
        }
    },
    "retr_trained_k1=1.0_b=0.9_w=1.0_fk=100": {
        "mrr@10": 0.3818169994726405,
        "recall@1": 0.2421430709368454,
        "recall@3": 0.4180634540556715,
        "recall@5": 0.4988027536665663,
        "recall@10": 0.5886710565698886,
        "recall@20": 0.6678389703681529,
        "ndcg@5": 0.3877554667477088,
        "ndcg@10": 0.41824283258468387,
        "ndcg@20": 0.4393097989832145,
        "success@100": 0.8525142173002095,
        "stage": "retrieval",
        "params": {
            "dense": "trained",
            "k1": 1.0,
            "b": 0.9,
            "

In [11]:
# Persist scores + cache to Drive (survives Colab reset)
!mkdir -p /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning
!cp -r results/hybrid-tuning/* /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning/

In [13]:
!cp results/hybrid-tuning/scores.json /content/drive/MyDrive/vnlegal-rag-v2/hybrid-tuning/scores.json

In [16]:
!mv results/hybrid-tuning/cache/* results/hybrid-tuning/

In [12]:
!git config user.email "vhphatdz@gmail.com"
!git config user.name "phatv1"
!git add results/hybrid-tuning/scores.json
!git commit -m "results: hybrid retrieval tuning (bm25+dense+cross sweep)" || echo "nothing to commit"
!git push || echo "push skipped"

[master c73fd78] results: hybrid retrieval tuning (bm25+dense+cross sweep)
 1 file changed, 1476 insertions(+)
 create mode 100644 results/hybrid-tuning/scores.json
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 12 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 7.57 KiB | 7.57 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/hyperformancelabs/vnlegal-rag-v2.git
   c591451..c73fd78  master -> master
